# LangChain MCP Adapters - chạy từng bước
Chạy lần lượt từng cell từ trên xuống. Notebook dùng top-level `await`.

## Bước 1: Cài đặt và import thư viện

In [12]:
# Cài vào đúng môi trường của kernel (chỉ cần chạy 1 lần, sau đó restart kernel)
# --only-binary=cryptography: tránh build cryptography từ source (cần Rust, hay lỗi)
%pip install -q -U --only-binary=cryptography langchain-mcp-adapters langgraph "langchain[openai]" mcp python-dotenv

import asyncio
import json
import os
import warnings

from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

Note: you may need to restart the kernel to use updated packages.


## Bước 2: Cấu hình môi trường và hằng số
Cần `OPENAI_API_KEY` trong file `.env`.

In [13]:
warnings.filterwarnings("ignore")
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Thiếu OPENAI_API_KEY"

MODEL = "openai:gpt-4.1"
MATH_SERVER = os.path.abspath("math_server.py")
WEATHER_SERVER = os.path.abspath("weather_server.py")
WEATHER_PORT = 8000
STREAMABLE_PORT = 3000
QUESTION = "what's (3 + 5) x 12?"

## Bước 3: Tạo Math MCP Server (stdio)

In [14]:
import os

# Phòng trường hợp chưa chạy Bước 2
if "MATH_SERVER" not in globals():
    MATH_SERVER = os.path.abspath("math_server.py")

math_code = '''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


@mcp.tool()
def divide(a: int, b: int) -> float:
    """Divide a by b (fails when b == 0)"""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


@mcp.prompt()
def math_helper(problem: str) -> str:
    """Prompt giải toán"""
    return f"Solve step by step using the math tools: {problem}"


@mcp.resource("config://math")
def math_config() -> str:
    """Math server config"""
    return "precision=2"


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open(MATH_SERVER, "w") as f:
    f.write(math_code)
print("Wrote", MATH_SERVER)

Wrote /Users/macbookairm1/Desktop/AI/AI/udemy/master-rag-course-demos/07-langchain-mcp-adapters/math_server.py


## Bước 4: Tạo Weather MCP Server (http)
Sau khi ghi file, chạy server ở terminal riêng: `python weather_server.py`
(hoặc chạy nền bằng cell bên dưới).

In [20]:
import os
import subprocess
import sys
import time

# Phòng trường hợp chưa chạy Bước 2
if "WEATHER_PORT" not in globals():
    WEATHER_PORT = 8000
if "WEATHER_SERVER" not in globals():
    WEATHER_SERVER = os.path.abspath("weather_server.py")

weather_code = f'''import logging

from mcp.server.fastmcp import FastMCP

# Ẩn ClosedResourceError vô hại khi client đóng session
logging.getLogger("mcp.server.streamable_http").setLevel(logging.CRITICAL)

mcp = FastMCP("Weather", port={WEATHER_PORT}, log_level="WARNING")


@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''
with open(WEATHER_SERVER, "w") as f:
    f.write(weather_code)

# Chạy nền (bỏ qua nếu đã chạy ở terminal khác)
weather_proc = subprocess.Popen([sys.executable, WEATHER_SERVER])
time.sleep(3)
print("Weather server pid:", weather_proc.pid)
# Dừng khi xong: weather_proc.terminate()


ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): [errno 48] address already in use


ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): [errno 48] address already in use


Weather server pid: 82332


## Bước 5: Kết nối một MCP server qua stdio

In [22]:
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=[MATH_SERVER])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print("Server:", init.serverInfo.name, "| protocol:", init.protocolVersion)

Server: Math | protocol: 2025-11-25


## Bước 6: Load MCP tools và chạy LangGraph agent

In [23]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)

        for t in tools:
            print("Tool:", t.name, "-", t.description)
            print("Schema:", json.dumps(t.args, ensure_ascii=False))

        agent = create_agent(MODEL, tools)
        result = await agent.ainvoke({"messages": QUESTION})
        print("\nAnswer:", result["messages"][-1].content)

Tool: add - Add two numbers
Schema: {"a": {"title": "A", "type": "integer"}, "b": {"title": "B", "type": "integer"}}
Tool: multiply - Multiply two numbers
Schema: {"a": {"title": "A", "type": "integer"}, "b": {"title": "B", "type": "integer"}}
Tool: divide - Divide a by b (fails when b == 0)
Schema: {"a": {"title": "A", "type": "integer"}, "b": {"title": "B", "type": "integer"}}

Answer: (3 + 5) equals 8. Then 8 multiplied by 12 is 96. So, (3 + 5) × 12 = 96.

Answer: (3 + 5) equals 8. Then 8 multiplied by 12 is 96. So, (3 + 5) × 12 = 96.


## Bước 7: Nhiều MCP server với MultiServerMCPClient

In [24]:
client = MultiServerMCPClient(
    {
        "math": {
            "command": sys.executable,
            "args": [MATH_SERVER],
            "transport": "stdio",
        },
        "weather": {
            "url": f"http://localhost:{WEATHER_PORT}/mcp",
            "transport": "http",
        },
    }
)
tools = await client.get_tools()
print([t.name for t in tools])

agent = create_agent(MODEL, tools)
math_response = await agent.ainvoke({"messages": QUESTION})
print("Math:", math_response["messages"][-1].content)
weather_response = await agent.ainvoke({"messages": "what is the weather in nyc?"})
print("Weather:", weather_response["messages"][-1].content)

Created new transport with session ID: da1862e93d1d4feaa38c9b08aebcbd70
Processing request of type ListToolsRequest
Terminating session: da1862e93d1d4feaa38c9b08aebcbd70
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:62911 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62914 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62915 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62917 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62919 - "DELETE /mcp HTTP/1.1" 200 OK
['add', 'multiply', 'divide', 'get_weather']
['add', 'multiply', 'divide', 'get_weather']
Math: (3 + 5) x 12 equals 96.
Math: (3 + 5) x 12 equals 96.


Created new transport with session ID: da1862e93d1d4feaa38c9b08aebcbd70
Processing request of type ListToolsRequest
Terminating session: da1862e93d1d4feaa38c9b08aebcbd70
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:62911 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62914 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62915 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62917 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62919 - "DELETE /mcp HTTP/1.1" 200 OK
['add', 'multiply', 'divide', 'get_weather']
['add', 'multiply', 'divide', 'get_weather']
Math: (3 + 5) x 12 equals 96.
Math: (3 + 5) x 12 equals 96.


Created new transport with session ID: f8bbb62e1aae47cc97537363f6226236
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Terminating session: f8bbb62e1aae47cc97537363f6226236
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:62925 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62928 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62929 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62931 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62933 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62935 - "DELETE /mcp HTTP/1.1" 200 OK
Weather: According to the information I have, it's always sunny in New York City. If you need more detailed or current weather data, let me know!
Weather: According to the information I have, it's always sunny in New York City. If you need more detailed or current weather data, let me know!


## Bước 8: Dùng explicit session với load_mcp_tools
Mặc định mỗi lần gọi tool sẽ tạo `ClientSession` mới. Dùng `client.session(...)` để tái sử dụng một session.

In [25]:
async with client.session("math") as session:
    tools = await load_mcp_tools(session)
    agent = create_agent(MODEL, tools)
    result = await agent.ainvoke({"messages": QUESTION})
    print(result["messages"][-1].content)

(3 + 5) x 12 is 96.


## Bước 9: Streamable HTTP transport
Cần server chạy ở cổng 3000, ví dụ:
`cd examples/servers/streamable-http-stateless && uv run mcp-simple-streamablehttp-stateless --port 3000`
(hoặc dùng FastMCP với `transport="streamable-http"`). Cell này sẽ lỗi kết nối nếu chưa có server.

In [27]:
from mcp.client.streamable_http import streamablehttp_client

STREAMABLE_URL = f"http://localhost:{STREAMABLE_PORT}/mcp"

# Cách 1: dùng trực tiếp streamablehttp_client
try:
    async with streamablehttp_client(STREAMABLE_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_agent(MODEL, tools)
            r = await agent.ainvoke({"messages": QUESTION})
            print("Direct:", r["messages"][-1].content)
except Exception as e:
    print("Streamable server chưa chạy:", type(e).__name__)

# Cách 2: qua MultiServerMCPClient
try:
    http_client = MultiServerMCPClient({"math": {"transport": "http", "url": STREAMABLE_URL}})
    tools = await http_client.get_tools()
    agent = create_agent(MODEL, tools)
    r = await agent.ainvoke({"messages": QUESTION})
    print("Multi:", r["messages"][-1].content)
except Exception as e:
    print("Streamable server chưa chạy:", type(e).__name__)

Streamable server chưa chạy: ExceptionGroup
Streamable server chưa chạy: ExceptionGroup


## Bước 10: Truyền runtime headers
Chỉ hỗ trợ transport `sse` và `http`. Header được gửi kèm mọi HTTP request.

In [31]:
header_client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": f"http://localhost:{WEATHER_PORT}/mcp",
            "headers": {
                "Authorization": "Bearer YOUR_TOKEN",
                "X-Custom-Header": "custom-value",
            },
        }
    }
)
tools = await header_client.get_tools()
agent = create_agent(MODEL, tools)
response = await agent.ainvoke({"messages": "what is the weather in HCM city?"})
print(response["messages"][-1].content)

Created new transport with session ID: f5a903f2bdd14950a503fafddaec7e8d
Processing request of type ListToolsRequest
Terminating session: f5a903f2bdd14950a503fafddaec7e8d
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:63036 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63039 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:63040 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63042 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63044 - "DELETE /mcp HTTP/1.1" 200 OK


Created new transport with session ID: f5a903f2bdd14950a503fafddaec7e8d
Processing request of type ListToolsRequest
Terminating session: f5a903f2bdd14950a503fafddaec7e8d
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:63036 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63039 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:63040 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63042 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63044 - "DELETE /mcp HTTP/1.1" 200 OK


Created new transport with session ID: f4b4cfc6c14249d1b20c1b402497346a
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Terminating session: f4b4cfc6c14249d1b20c1b402497346a
Error in standalone SSE writer
Traceback (most recent call last):
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/mcp/server/streamable_http.py", line 765, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/abc/_streams.py", line 37, in __anext__
    return await self.receive()
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/macbookairm1/opt/anaconda3/envs/rag-course/lib/python3.12/site-packages/anyio/streams/memory.py", line 141, in receive
    raise ClosedResourceError from None
anyio.ClosedResourceError


INFO:     127.0.0.1:63047 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63050 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63051 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:63053 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63055 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:63057 - "DELETE /mcp HTTP/1.1" 200 OK
It seems I received incorrect data for your weather request. Would you like me to try again or provide general weather information for Ho Chi Minh City? Typically, HCM City has a tropical climate with hot and humid weather year-round, and June usually brings plenty of rain. Let me know if you'd like a more precise update!
It seems I received incorrect data for your weather request. Would you like me to try again or provide general weather information for Ho Chi Minh City? Typically, HCM City has a tropical climate with hot and humid weather year-round, and June usually brings plenty of rain. Let me know if you'd like a more precise update!


## Bước 11: Xử lý lỗi tool
- `handle_tool_errors=True` (mặc định): lỗi thực thi trả về `ToolMessage` với `status="error"`, agent có thể tự sửa.
- `handle_tool_errors=False`: raise `ToolException`.

Dùng tool `divide` (chia cho 0 sẽ lỗi).

In [29]:
math_conn = {"math": {"command": sys.executable, "args": [MATH_SERVER], "transport": "stdio"}}

# Mặc định: handle_tool_errors=True
safe_client = MultiServerMCPClient(math_conn)
safe_tools = await safe_client.get_tools()
divide_tool = next(t for t in safe_tools if t.name == "divide")
res = await divide_tool.ainvoke({"a": 1, "b": 0})
print("handle_tool_errors=True ->", res)

safe_agent = create_agent(MODEL, safe_tools)
r = await safe_agent.ainvoke({"messages": "What is 10 divided by 0? Explain."})
for m in r["messages"]:
    if m.type == "tool":
        print("ToolMessage status:", m.status)
print(r["messages"][-1].content)

# Legacy: handle_tool_errors=False
strict_client = MultiServerMCPClient(math_conn, handle_tool_errors=False)
strict_tools = await strict_client.get_tools()
strict_divide = next(t for t in strict_tools if t.name == "divide")
try:
    await strict_divide.ainvoke({"a": 1, "b": 0})
except Exception as e:
    print("handle_tool_errors=False ->", type(e).__name__, e)

handle_tool_errors=True -> [{'type': 'text', 'text': 'Error executing tool divide: Cannot divide by zero', 'id': 'lc_f240e28c-d1a4-413c-936a-c4dd8e11312d'}]
Dividing 10 by 0 is mathematically undefined. Here’s why:

- **Division** is the process of finding out how many times one number (the divisor) fits into another number (the dividend).
- If you try to divide 10 by 0, you are essentially trying to figure out: "How many times does 0 fit into 10?"
- But multiplying any number by 0 always results in 0, never 10. There is no real number that you can multiply by 0 to get 10.
- That’s why division by zero is **undefined** in mathematics. In other words, there is no answer.
Dividing 10 by 0 is mathematically undefined. Here’s why:

- **Division** is the process of finding out how many times one number (the divisor) fits into another number (the dividend).
- If you try to divide 10 by 0, you are essentially trying to figure out: "How many times does 0 fit into 10?"
- But multiplying any num

## Bước 12: Tự dựng LangGraph StateGraph với MCP tools

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition

model = init_chat_model(MODEL)
tools = await client.get_tools()  # math + weather


def call_model(state: MessagesState):
    response = model.bind_tools(tools).invoke(state["messages"])
    return {"messages": response}


builder = StateGraph(MessagesState)
builder.add_node(call_model)
builder.add_node(ToolNode(tools))
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", tools_condition)
builder.add_edge("tools", "call_model")
graph = builder.compile()

r1 = await graph.ainvoke({"messages": QUESTION})
print(r1["messages"][-1].content)
r2 = await graph.ainvoke({"messages": "what is the weather in nyc?"})
print(r2["messages"][-1].content)

## Bước 13: Load MCP prompts, resources và server info

In [ ]:
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.resources import load_mcp_resources, get_mcp_resource
from langchain_mcp_adapters.server_info import load_mcp_server_info

async with client.session("math") as session:
    # Prompt -> LangChain messages
    messages = await load_mcp_prompt(session, "math_helper", arguments={"problem": "2 + 2"})
    print("Prompt messages:", messages)

    # Resources -> Blob
    blobs = await load_mcp_resources(session)
    for b in blobs:
        print("Resource:", b.source, "|", b.as_string())
    one = await get_mcp_resource(session, "config://math")
    print("Single resource:", [b.as_string() for b in one])

    # Server info (initialize handshake)
    info = await load_mcp_server_info(session)
    print("Server info:", info)

## Bước 14: Tool call interceptors và callbacks
Interceptor bọc quá trình gọi tool (logging, retry, cache...). Callbacks nhận log/progress từ server.

In [ ]:
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext
from langchain_mcp_adapters.interceptors import MCPToolCallRequest


async def logging_interceptor(request: MCPToolCallRequest, handler):
    print(f"[interceptor] {request.server_name}.{request.name} args={request.args}")
    for attempt in range(2):  # retry đơn giản
        try:
            result = await handler(request)
            print("[interceptor] done")
            return result
        except Exception as e:
            print(f"[interceptor] attempt {attempt + 1} failed: {e}")
            if attempt == 1:
                raise


async def on_logging(params, context: CallbackContext):
    print(f"[server log:{context.server_name}]", params.data)


async def on_progress(progress, total, message, context: CallbackContext):
    print(f"[progress:{context.server_name}] {progress}/{total} {message or ''}")


intercepted_client = MultiServerMCPClient(
    math_conn,
    callbacks=Callbacks(on_logging_message=on_logging, on_progress=on_progress),
    tool_interceptors=[logging_interceptor],
)
tools = await intercepted_client.get_tools()
agent = create_agent(MODEL, tools)
r = await agent.ainvoke({"messages": QUESTION})
print(r["messages"][-1].content)

## Bước 15: Expose agent qua LangGraph API Server
Lưu ý: stdio được thiết kế cho ứng dụng chạy trên máy người dùng; cân nhắc trước khi dùng trong web server.

In [ ]:
graph_code = f'''from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent


async def make_graph():
    client = MultiServerMCPClient(
        {{
            "weather": {{
                "url": "http://localhost:{WEATHER_PORT}/mcp",
                "transport": "http",
            }},
            "math": {{
                "command": "python",
                "args": [{MATH_SERVER!r}],
                "transport": "stdio",
            }},
        }}
    )
    tools = await client.get_tools()
    return create_agent("{MODEL}", tools)
'''
with open("graph.py", "w") as f:
    f.write(graph_code)

with open("langgraph.json", "w") as f:
    json.dump({"dependencies": ["."], "graphs": {"agent": "./graph.py:make_graph"}}, f, indent=2)

print(graph_code)
# Chạy server: langgraph dev